In [ ]:
from concurrent.futures import ProcessPoolExecutor
import json
from pathlib import Path
import sys
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path("..").resolve()))

from _simulation_workers import (
    generate_scenario1,
    generate_scenario2,
    generate_scenario3,
    init_globals,
    sweep_worker,
)

from geomexp.visualization import ClusterVisualizer, PlotStyle

warnings.filterwarnings("ignore")

_plasma = mpl.colormaps["plasma"]
_plasma_colors = [mpl.colors.to_hex(_plasma(i / 4)) for i in range(5)]

style = PlotStyle(figsize=(3, 4), use_latex=True, fontsize=16, color_palette=_plasma_colors)
viz = ClusterVisualizer(style)

PLOT_DIR = Path("../plots/simulation")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = Path("../results/simulation")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

N_MC = 50
N_RESTARTS = 10
K = 3
MAX_WORKERS = 8

init_globals(K, N_RESTARTS)

In [ ]:
def run_sweep_study(generate_fn, param_name, param_values, n_mc=N_MC, base_seed=0):
    """Run a full MC study sweeping one DGP parameter."""
    all_args = []
    for val in param_values:
        kwargs = {param_name: val}
        for mc in range(n_mc):
            all_args.append((base_seed + mc, generate_fn, kwargs, val))

    with ProcessPoolExecutor(
        max_workers=MAX_WORKERS,
        initializer=init_globals,
        initargs=(K, N_RESTARTS),
    ) as pool:
        raw_results = list(pool.map(sweep_worker, all_args))

    grouped = {}
    for val, res in raw_results:
        grouped.setdefault(val, []).append(res)

    summary = {}
    for val in param_values:
        reps = grouped[val]
        s = {}
        for method in ["kmeans", "gec"]:
            metrics = {}
            for key in ["ari", "vi", "miscl", "runtime"]:
                vals_list = [r[method][key] for r in reps]
                metrics[key] = {"mean": float(np.mean(vals_list)),
                                "std": float(np.std(vals_list))}
            s[method] = metrics
        s["selected_r"] = {
            "mean": float(np.mean([r["selected_r"] for r in reps])),
            "std": float(np.std([r["selected_r"] for r in reps])),
        }
        summary[val] = s

    return summary, grouped


def print_sweep_summary(name, param_label, summary):
    print(f"\n{'='*70}")
    print(f"  {name} -- sweep over {param_label}")
    print(f"{'='*70}")
    for val in sorted(summary.keys()):
        s = summary[val]
        sel_r = s["selected_r"]
        print(f"\n  {param_label} = {val}")
        print(f"  Selected r: {sel_r['mean']:.3f} ({sel_r['std']:.3f})")
        print(f"  {'Metric':<15} {'k-means':>20} {'GEC':>20}")
        print(f"  {'-'*55}")
        for key, label in [("ari", "ARI"), ("vi", "VI"),
                           ("miscl", "Miscl. error"), ("runtime", "Runtime (s)")]:
            km = s["kmeans"][key]
            gc = s["gec"][key]
            print(f"  {label:<15} {km['mean']:>8.3f} ({km['std']:.3f})"
                  f" {gc['mean']:>8.3f} ({gc['std']:.3f})")

In [ ]:
def plot_ari_sweep(summary, param_label, filename):
    """Plot ARI vs swept parameter for k-means and GEC."""
    vals = sorted(summary.keys())
    km_mean = [summary[v]["kmeans"]["ari"]["mean"] for v in vals]
    km_std = [summary[v]["kmeans"]["ari"]["std"] for v in vals]
    gec_mean = [summary[v]["gec"]["ari"]["mean"] for v in vals]
    gec_std = [summary[v]["gec"]["ari"]["std"] for v in vals]

    fig, ax = plt.subplots(figsize=style.figsize)
    ax.errorbar(vals, km_mean, yerr=km_std, label="k-means", marker="o",
                markersize=3, linewidth=1, capsize=2, color=style.color_palette[0])
    ax.errorbar(vals, gec_mean, yerr=gec_std, label="GEC", marker="s",
                markersize=3, linewidth=1, capsize=2, color=style.color_palette[1])
    ax.set_xlabel(param_label)
    ax.set_ylabel(r"ARI")
    ax.legend(frameon=True, edgecolor="black", fancybox=False)
    ax.grid(True, alpha=style.grid_alpha, linewidth=style.grid_linewidth)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / filename, bbox_inches="tight")
    plt.show()
    return fig

In [ ]:
def fmt(mean, std, bold=False):
    s = f"{mean:.3f} ({std:.3f})"
    return rf"\textbf{{{s}}}" if bold else s


METRIC_LABELS = [
    ("ari", "ARI", True),
    ("vi", "VI", False),
    ("miscl", "Miscl.\\ error", False),
    ("runtime", "Runtime (s)", False),
]


def _bold_flags(key, km_mean, gc_mean, higher_better):
    if key == "runtime":
        return False, False
    if higher_better:
        return km_mean > gc_mean, gc_mean > km_mean
    return km_mean < gc_mean, gc_mean < km_mean


def latex_combined_table(scenario_studies):
    """Generate one LaTeX table combining all scenarios.

    Parameters
    ----------
    scenario_studies : list of (scenario_name, param_latex, summary) tuples.
        Each summary is {param_value: {method: {metric: {mean, std}}, ...}}.
    """
    caption = (
        r"Simulation results across three DGPs. Entries are means (std) over "
        f"{N_MC} Monte Carlo replications. "
        r"Best values per metric are in \textbf{bold}."
    )
    lines = [
        r"\begin{table}[ht]",
        r"\centering",
        r"\small",
        rf"\caption{{{caption}}}",
        r"\label{tab:simulation}",
        r"\begin{tabular}{ll l rr}",
        r"\toprule",
        r"Scenario & Setting & Metric & k-means & GEC \\",
        r"\midrule",
    ]
    for sc_idx, (sc_name, param_latex, summary) in enumerate(scenario_studies):
        vals = sorted(summary.keys())
        for vi, val in enumerate(vals):
            s = summary[val]
            for j, (key, label, higher_better) in enumerate(METRIC_LABELS):
                km, gc = s["kmeans"][key], s["gec"][key]
                km_b, gc_b = _bold_flags(
                    key, km["mean"], gc["mean"], higher_better,
                )
                sc_col = sc_name if vi == 0 and j == 0 else ""
                setting = f"${param_latex}={val}$" if j == 0 else ""
                lines.append(
                    f"  {sc_col} & {setting} & {label} & "
                    f"{fmt(km['mean'], km['std'], km_b)} & "
                    f"{fmt(gc['mean'], gc['std'], gc_b)} \\\\"
                )
            if vi < len(vals) - 1:
                lines.append(r"  \addlinespace")
        if sc_idx < len(scenario_studies) - 1:
            lines.append(r"\midrule")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    print("\n".join(lines))

In [ ]:
rng_plot = np.random.default_rng(0)
t_plot = np.linspace(0, 1, 100)

scenarios = [
    ("Scenario 1", generate_scenario1, {}),
    ("Scenario 2", generate_scenario2, {}),
    ("Scenario 3", generate_scenario3, {}),
]

style.fontsize = 15
viz._setup_matplotlib()
fig, axes = plt.subplots(len(scenarios), K, figsize=(11, 10), sharey=True, sharex=True)

for row, (title, gen_fn, kwargs) in enumerate(scenarios):
    X_ex, y_ex = gen_fn(seed=69, **kwargs)
    for k in range(K):
        ax = axes[row, k]
        mask = y_ex == k
        color = style.color_palette[k % len(style.color_palette)]

        cluster_curves = X_ex[mask]
        mean_curve = cluster_curves.mean(axis=0)
        q1 = np.percentile(cluster_curves, 25, axis=0)
        q3 = np.percentile(cluster_curves, 75, axis=0)

        # Individual curves with low opacity
        for curve in cluster_curves:
            ax.plot(t_plot, curve, color=color, alpha=0.1, linewidth=0.4)

        # IQR shaded band
        ax.fill_between(t_plot, q1, q3, color=color, alpha=0.2)

        # Mean curve
        ax.plot(t_plot, mean_curve, color=color, linewidth=2)

        # Dashed reference line at zero
        ax.axhline(0, color="darkgrey", linewidth=0.75, linestyle="--", alpha=0.8)

        if row == 0:
            ax.set_title(f"Cluster {k + 1}")
        if k == 0:
            ax.set_ylabel(title)
        else:
            ax.tick_params(axis="y", which="both", left=False, labelleft=False)

        ax.set_ylim(-5, 8)
        ax.set_yticks([-4, -2, 0, 2, 4, 6, 8])

        # Clean: no gridlines, minimal spines
        ax.grid(False)

        if row < len(scenarios) - 1:
            ax.tick_params(axis="x", labelbottom=False, length=0)


fig.subplots_adjust(left=0.1, wspace=0.0, hspace=0.0)
fig.supxlabel("$t$", y=0.03)
fig.supylabel("$X(t)$", x=0.03)
plt.tight_layout()
fig.savefig(PLOT_DIR / "dgp_overview.pdf", bbox_inches="tight")
plt.show()

In [ ]:
sigma_eps_values = np.round(np.arange(0.1, 1.4, 0.3), decimals=1)
summary1, grouped1 = run_sweep_study(
    generate_scenario1, "sigma_eps", sigma_eps_values, base_seed=1000
)
print_sweep_summary("Scenario 1", r"$\sigma_\varepsilon$", summary1)

In [ ]:
# Sweep tau (tail strength) with R=2.0, sigma_c=0.1
tau_values_s2 = np.arange(0.2, 1.5, 0.3)
summary2_tau, grouped2_tau = run_sweep_study(
    generate_scenario2, "tau", tau_values_s2, base_seed=2000
)
print_sweep_summary("Scenario 2 (tau sweep)", r"$\tau$", summary2_tau)

In [ ]:
tau_values_s3 = np.arange(0.2, 1.5, 0.3)
summary3, grouped3 = run_sweep_study(
    generate_scenario3, "tau", tau_values_s3, base_seed=3000
)
print_sweep_summary("Scenario 3", r"$\tau$", summary3)

In [ ]:
def _convert_keys(summary):
    """Convert numeric keys to strings for JSON serialization."""
    return {str(k): v for k, v in summary.items()}

def _restore_keys(summary):
    """Restore string keys back to floats."""
    return {float(k): v for k, v in summary.items()}

with open(RESULTS_DIR / "scenario1_sweep.json", "w") as f:
    json.dump(_convert_keys(summary1), f, indent=2)
with open(RESULTS_DIR / "scenario2_tau_sweep.json", "w") as f:
    json.dump(_convert_keys(summary2_tau), f, indent=2)
with open(RESULTS_DIR / "scenario3_sweep.json", "w") as f:
    json.dump(_convert_keys(summary3), f, indent=2)

In [ ]:
with open(RESULTS_DIR / "scenario1_sweep.json") as f:
    summary1 = _restore_keys(json.load(f))
with open(RESULTS_DIR / "scenario2_tau_sweep.json") as f:
    summary2_tau = _restore_keys(json.load(f))
with open(RESULTS_DIR / "scenario3_sweep.json") as f:
    summary3 = _restore_keys(json.load(f))

In [ ]:
latex_combined_table([
    ("Scenario 1", r"\sigma_\varepsilon", summary1),
    ("Scenario 2", r"\tau", summary2_tau),
    ("Scenario 3", r"\tau", summary3),
])